# Deep Agents, Skills, and AGENTS.md

A deep agent's system prompt is typically static, whereas the things you want it to know are not.
**Skills** and **AGENTS.md (a form of memory)** are the two answers, and they work in opposite ways:

- A **skill** is a folder of instructions the agent loads *only when a task calls for it*.
- **Memory** (<code style="font-size:15px; color:#2b8a3e;">AGENTS.md</code>) is text injected into *every* prompt.

We'll build both for a fictional on-call assistant at **Acme**, a small SaaS team — a supervisor that handles the general on-call load, and a **triage specialist** it delegates alerts to. The specialist carries its own runbook, which the supervisor never sees.


In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(override=True)

from util import print_activity, print_exchange, show_file, show_skills, show_tree
from deepagents import MemoryMiddleware, create_deep_agent
from deepagents.backends import FilesystemBackend
from langgraph.checkpoint.memory import MemorySaver

model = "claude-haiku-4-5-20251001"
oncall_prompt = "You are the on-call assistant for Acme, a small SaaS team."

# Everything the agent knows lives under oncall_home/. virtual_mode=True makes that directory its entire filesystem:
# the paths it sees ("/skills/supervisor/") are virtual and rooted here.
ONCALL_HOME = str((Path.cwd() / "oncall_home").resolve())
backend = FilesystemBackend(root_dir=ONCALL_HOME, virtual_mode=True)

# Skills and AGENTS.md

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;" width="16%"></th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Skills</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">AGENTS.md</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Loaded</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Name and description at startup; the rest only when the agent decides it applies</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">In full, into every prompt, every turn</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Format</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="font-size:15px; color:#2b8a3e;">SKILL.md</code> in a named directory, plus any supporting files</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="font-size:15px; color:#2b8a3e;">AGENTS.md</code> — plain markdown, no required structure</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Use for</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Procedures — detailed, task-specific, possibly long. A runbook.</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Conventions — short, and true on every single task.</td>
</tr>
</tbody>
</table>

For our fictional on-call assistant the split between **skills** and **AGENTS.md** is clear:
<ul>
    <li><b>How to triage a database alert</b> is a runbook nobody needs until the pager goes off. This is a perfect fit for <b>progressive disclosure</b>.</li>
    <li>On the other hand, knowing how to properly classify an alert as while <b>SEV1/SEV2/SEV3</b> is critical for every message the agent ever writes.</li>
</ul>


# Progressive disclosure

Skills are cheap because the agent doesn't read them until it needs them. This is called <i><b>progressive disclosure</b></i>. The idea is that <b>the context window is a precious, limited resource</b> - it shouldn't be overloaded with things it may never need.

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;" width="20%">Level</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">What enters context</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Loaded into context when</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>1 — Metadata</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The <code style="font-size:15px; color:#2b8a3e;">name</code> and <code style="font-size:15px; color:#2b8a3e;">description</code> from each skill's YAML frontmatter</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Startup, for every skill</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>2 — Instructions</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The full <code style="font-size:15px; color:#2b8a3e;">SKILL.md</code> body</td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">When the agent matches a task to a description</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>3 — Resources</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Files under <code style="font-size:15px; color:#2b8a3e;">references/</code>, <code style="font-size:15px; color:#2b8a3e;">scripts/</code>, <code style="font-size:15px; color:#2b8a3e;">assets/</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Only if the level-2 instructions point at them</td>
</tr>
</tbody>
</table>

A skill's standing cost to the **context window** is two lines, whatever its size.
<br/>
The built-in <code style="font-size:15px; color:#2b8a3e;">SkillsMiddleware</code> does levels 1 and 2; level 3 is just the model calling <code style="font-size:15px; color:#2b8a3e;">read_file</code> on paths the skill told it about.

# What it looks like on disk

Two skill sources, split by **who reads them**: the supervisor is granted <code style="font-size:15px; color:#2b8a3e;">skills/supervisor/</code>, while <code style="font-size:15px; color:#2b8a3e;">skills/triage/</code> belongs to the specialist alone. Neither agent sees the other's. Within a source, a skill's own directory name (<code style="font-size:15px; color:#2b8a3e;">postmortem/</code>, <code style="font-size:15px; color:#2b8a3e;">triage-alert/</code>) **must** match the <code style="font-size:15px; color:#2b8a3e;">name</code> in its frontmatter.


In [13]:
show_tree(ONCALL_HOME)

```
oncall_home/
├── memory/
│   └── notes.md
├── skills/
│   ├── supervisor/
│   │   └── postmortem/
│   │       └── SKILL.md
│   └── triage/
│       └── triage-alert/
│           ├── references/
│           │   └── severity-matrix.md
│           └── SKILL.md
└── AGENTS.md
```

# Skills: Markdown with frontmatter

Here is <code style="font-size:15px; color:#2b8a3e;">triage-alert</code> in full — the specialist's whole job description. The frontmatter is all its agent sees at startup; everything below it is a level 2 disclosure.

Two details are doing the real work:

1. The **description says what *and* when**, in words a request might actually use — "alert", "pager", "how bad is this?".
2. Step 1 sends the agent to <code style="font-size:15px; color:#2b8a3e;">references/severity-matrix.md</code> and tells it severity is decided there, *never by feel*. That's a deliberate level-3 trigger.


In [14]:
show_file(f"{ONCALL_HOME}/skills/triage/triage-alert/SKILL.md")

**`/Users/andrew/src/langchain-demos/deepagents-deep-dive/oncall_home/skills/triage/triage-alert/SKILL.md`**

<div class="highlight" style="background: #ffffff; background:#ffffff !important; color:#1a1a1a !important; border:1px solid #d0d7de; border-radius:6px; padding:12px 14px; margin:8px 0; overflow-x:auto;"><pre style="background:transparent !important; color:#1a1a1a !important; margin:0; line-height:1.45;; line-height: 125%;"><span></span>---
name: triage-alert
description: &gt;-
  Triage a production alert or pager notification: assign a severity, state the
  blast radius, and give the first actions. Use when handed an alert, a pager
  message, an error-rate or latency spike, or asked &quot;how bad is this?&quot;.
---

# triage-alert

## Overview

Turns a raw alert into a severity, a blast radius, and the first three things to do.

## Instructions

<span style="color: #A90D91">1.</span> **Read `references/severity-matrix.md` first.** Severity is decided by the
   matrix, never by feel. The matrix also carries override rules that beat the
   error-rate rows outright — you cannot get severity right without reading them.
<span style="color: #A90D91">2.</span> State the **blast radius**: what share of traffic or which customers are
   affected. If the alert doesn&#39;t say, write <span style="color: #C41A16">`unknown`</span>.
<span style="color: #A90D91">3.</span> Assign the severity and **quote the matrix row or rule you matched**.
<span style="color: #A90D91">4.</span> Give the **first three actions**, most reversible first.
<span style="color: #A90D91">5.</span> State whether a status-page update is required, per the matrix.

## Output format

<span style="color: #C41A16">```</span>
<span style="color: #C41A16">Severity: SEV&lt;n&gt; — matched: &lt;the row or rule you applied&gt;</span>
<span style="color: #C41A16">Blast radius: &lt;share of traffic or customers, or &quot;unknown&quot;&gt;</span>
<span style="color: #C41A16">Status page: &lt;required within N min | not required&gt;</span>
<span style="color: #C41A16">First actions:</span>
<span style="color: #C41A16">  1. &lt;most reversible&gt;</span>
<span style="color: #C41A16">  2. ...</span>
<span style="color: #C41A16">  3. ...</span>
<span style="color: #C41A16">```</span>

## Edge cases

<span style="color: #A90D91">-</span> Symptom sits between two rows → round up, never down.
<span style="color: #A90D91">-</span> Several symptoms at once → triage the worst one, and note the others.
<span style="color: #A90D91">-</span> No error rate given → say so and triage on the remaining evidence.
</pre></div>


# Subagents get their own skills

Triage is a good candidate for a **subagent**: it has a long runbook, a reference file, and a fixed output format — none of which the supervisor needs while it is doing anything else.

Subagents **do not inherit skills**. A subagent starts with an empty catalog and gets exactly the sources you list in its spec. So the split is enforced by the two <code style="font-size:15px; color:#2b8a3e;">skills=[...]</code> lists, not by convention:

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;" width="24%"></th>
<th style="border:1px solid #999; padding:10px; text-align:left;">supervisor</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">triage-specialist</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="font-size:15px; color:#2b8a3e;">skills</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="font-size:15px; color:#2b8a3e;">/skills/supervisor/</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="font-size:15px; color:#2b8a3e;">/skills/triage/</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Progressive disclosure level 1</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="font-size:15px; color:#2b8a3e;">postmortem</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="font-size:15px; color:#2b8a3e;">triage-alert</code></td>
</tr>
</tbody>
</table>

What the supervisor gets in exchange is **one line** in the `task` tool: the subagent's `description`. That line is the entire interface — same rule as a skill description, one level up. Write it in the words a request would use, because it is all the supervisor has to decide on.


In [15]:
triage_specialist = {
    "name": "triage-specialist",
    "description": (
        "Triage a production alert or pager notification. Hand it the raw alert text; it "
        "returns a severity, the blast radius, and the first actions. Use for any alert, "
        "pager message, error-rate or latency spike, or \"how bad is this?\"."
    ),
    "system_prompt": (
        "You are Acme's alert-triage specialist. Every alert you are handed is triaged with "
        "your triage-alert skill - follow it exactly, including the reference files it tells "
        "you to read. Return the skill's output format and nothing else."
    ),
    "skills": ["/skills/triage/"],
}

# The specialist's catalog. Nothing here is ever charged to the supervisor's context.
show_skills(backend, triage_specialist["skills"])

# Progressive Disclosure: Level 1 (injected at startup)

**Triage Skills**: `/skills/triage/` (higher priority)

**Available Skills:**

- **triage-alert**: Triage a production alert or pager notification: assign a severity, state the blast radius, and give the first actions. Use when handed an alert, a pager message, an error-rate or latency spike, or asked "how bad is this?".
  -> Read `/skills/triage/triage-alert/SKILL.md` for full instructions

# Progressive Disclosure: Levels 2 and 3

Now we will trigger a real page and watch progressive disclosure in action:
- First the supervisor invokes the specialist subagent
- Specialist subagent first loads its <code style="font-size:15px; color:#2b8a3e;">SKILL.md</code>
- Then the subagent loads <code style="font-size:15px; color:#2b8a3e;">severity-matrix.md</code>, which is _only_ referenced by its skill.

In [16]:
supervisor_prompt = (
    oncall_prompt + " Alerts and pages are not yours to triage - delegate them to the triage-specialist subagent and relay its answer verbatim."
)

supervisor = create_deep_agent(
    model=model,
    system_prompt=supervisor_prompt,
    backend=backend,
    skills=["/skills/supervisor/"],
    subagents=[triage_specialist],  # sub-agent defined above with its own skills
    checkpointer=MemorySaver(),
)

alert = """PagerDuty, 2026-07-27T14:32Z: db-primary replication lag 340s and climbing.
API error rate 0.7%. Writes are succeeding, but the read replica is serving stale data,
and the events-backfill job may have overwritten rows in the events table."""

triage_config = {"configurable": {"thread_id": "triage"}}
print_activity(
    supervisor,
    f"Triage this.\n\n{alert}",
    config=triage_config,
    title="Progressive disclosure timeline",
)

### Progressive disclosure timeline

- 🧑‍✈️ **supervisor** → 📨 delegates via **task** to `triage-specialist`
    - 🤖 **triage-specialist** → 🔧 **read_file**(`/skills/triage/triage-alert/SKILL.md`)
    - 🤖 **triage-specialist** ← 📥 `read_file` → 1 --- 2 name: triage-alert 3 description: >- 4 Triage a production alert or page…
    - 🤖 **triage-specialist** → 🔧 **read_file**(`/skills/triage/triage-alert/references/severity-matrix.md`)
    - 🤖 **triage-specialist** ← 📥 `read_file` → 1 # Severity matrix 2 3 ## Rows 4 5 | Condition | Severity | Status page | Who i…
- 🧑‍✈️ **supervisor** ← 📥 `task` → Now I'll apply the triage framework to the alert: **Alert Analysis:** The alert …

# AGENTS.md: Context that's always relevant (and always loaded)

Everything so far was conditional. But some things hold on every turn — UTC timestamps, `SEVn` severity names, never naming a person as the cause. Hiding those behind a skill would be wrong, because sometimes the agent wouldn't look.

That's memory. Point <code style="font-size:15px; color:#2b8a3e;">memory=[...]</code> at one or more <code style="font-size:15px; color:#2b8a3e;">AGENTS.md</code> files (the <a href="https://agents.md/">agents.md</a> convention) and they go into the system prompt on every request — no trigger, no <code style="font-size:15px; color:#2b8a3e;">read_file</code>, no chance of being skipped. The cost is symmetrical: you pay for it every turn, so it stays short.


In [17]:
show_file(f"{ONCALL_HOME}/AGENTS.md")

**`/Users/andrew/src/langchain-demos/deepagents-deep-dive/oncall_home/AGENTS.md`**

<div class="highlight" style="background: #ffffff; background:#ffffff !important; color:#1a1a1a !important; border:1px solid #d0d7de; border-radius:6px; padding:12px 14px; margin:8px 0; overflow-x:auto;"><pre style="background:transparent !important; color:#1a1a1a !important; margin:0; line-height:1.45;; line-height: 125%;"><span></span># Acme on-call conventions

&lt;!-- Curated by the team. The agent reads this every session and does not edit it.
     Things it learns go in /memory/notes.md instead. --&gt;

## The service

Acme is a small SaaS team running one API and one web app. Two engineers are on
call at a time: primary and secondary.

## How we write incidents

<span style="color: #A90D91">-</span> Timestamps in UTC, ISO 8601: <span style="color: #C41A16">`2026-07-27T14:32Z`</span>. Never local time, never &quot;2:32pm&quot;.
<span style="color: #A90D91">-</span> Severity is always <span style="color: #C41A16">`SEV1`</span> / <span style="color: #C41A16">`SEV2`</span> / <span style="color: #C41A16">`SEV3`</span>. Never &quot;critical&quot;, &quot;high&quot;, or &quot;P1&quot;.
<span style="color: #A90D91">-</span> Lead with impact, then cause, then action.

## Non-negotiables

<span style="color: #A90D91">-</span> Never invent customer-impact numbers. If you don&#39;t have them, write <span style="color: #C41A16">`unknown`</span>.
<span style="color: #A90D91">-</span> Never name an individual as the cause of an incident. Name the change, not the person.
<span style="color: #A90D91">-</span> Customer-facing messages are drafted for review, never sent.

## Where things live

<span style="color: #A90D91">-</span> Runbooks: <span style="color: #C41A16">`/skills/supervisor/`</span> (yours) and <span style="color: #C41A16">`/skills/triage/`</span> (the triage specialist&#39;s).
<span style="color: #A90D91">-</span> Learned preferences: <span style="color: #C41A16">`/memory/notes.md`</span>. When we correct you, append it there
  with <span style="color: #C41A16">`edit_file`</span> so it survives into later sessions.
</pre></div>


# Add AGENTS.md to the supervisor

In [18]:
summary_ask = (
    "Write the incident-channel summary for this. One short paragraph, then the severity line.\n\n"
    "At 2:32pm UTC on July 27 2026 the API began returning 502s for about 6% of requests. "
    "Dana deployed a connection-pool config change at 2:28pm that looks related.\n\n"
    "Reply inline - do not write any files."
)

with_memory = create_deep_agent(
    model=model,
    system_prompt=oncall_prompt,
    backend=backend,
    memory=["/AGENTS.md", "/memory/notes.md"],
)
print_exchange(with_memory.invoke(
    {"messages": [{"role": "user", "content": summary_ask}]},
    config={"configurable": {"thread_id": "summary"}},
))


### Human

Write the incident-channel summary for this. One short paragraph, then the severity line.

At 2:32pm UTC on July 27 2026 the API began returning 502s for about 6% of requests. Dana deployed a connection-pool config change at 2:28pm that looks related.

Reply inline - do not write any files.

### Ai

**Incident Summary**

Starting at 2026-07-27T14:32Z, the API experienced elevated error rates with approximately 6% of requests returning 502 errors. A connection-pool configuration change deployed at 2026-07-27T14:28Z appears to be the likely cause. Investigation and remediation are underway.

**Severity: SEV2**

# The agent's output is properly formatted according to the conventions specified in AGENTS.md

- Timestamp converted to ISO UTC
- Properly labeled 'SEV2'
- Engineer's name omitted

# Saving Memories: Making your agent smarter 🥸

<code style="font-size:15px; color:#2b8a3e;">MemoryMiddleware</code> doesn't just read files — it tells the agent they *are* its memory and that keeping them current is part of the job.

Give your agent a correction or new instruction and watch it update its memory.


In [19]:
learn_config = {"configurable": {"thread_id": "correction"}}
print_activity(
    with_memory,
    "One correction for future triages: start the first line with the severity in brackets, "
    "like '[SEV2] ...'. Our Slack search relies on that prefix. Apply it from now on.",
    config=learn_config,
    title="Agent update's its memory"
)
show_file(f"{ONCALL_HOME}/memory/notes.md")


### Agent update's its memory

- 🧑‍✈️ **supervisor** → 🔧 **edit_file**(`/memory/notes.md`)
- 🧑‍✈️ **supervisor** ← 📥 `edit_file` → Successfully replaced 1 instance(s) of the string in '/memory/notes.md'

**`/Users/andrew/src/langchain-demos/deepagents-deep-dive/oncall_home/memory/notes.md`**

<div class="highlight" style="background: #ffffff; background:#ffffff !important; color:#1a1a1a !important; border:1px solid #d0d7de; border-radius:6px; padding:12px 14px; margin:8px 0; overflow-x:auto;"><pre style="background:transparent !important; color:#1a1a1a !important; margin:0; line-height:1.45;; line-height: 125%;"><span></span># Learned preferences

<span style="color: #A90D91">-</span> **Incident reports**: Start the first line with severity in brackets like <span style="color: #C41A16">`[SEV2] ...`</span>. Slack search relies on this prefix for filtering.
</pre></div>


In [20]:
Path(f"{ONCALL_HOME}/memory/notes.md").write_text(
    "# Learned preferences\n\n- Nothing learned yet.\n"
)
print("notes.md reset")

notes.md reset


# Summary

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;" width="32%">Concept</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">What to remember</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="font-size:15px; color:#2b8a3e;">skills=["/skills/supervisor/"]</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Points at a directory <i>of</i> skill directories, each holding a <code style="font-size:15px; color:#2b8a3e;">SKILL.md</code>. Paths use forward slashes and are relative to the backend root. Directory name must equal the frontmatter <code style="font-size:15px; color:#2b8a3e;">name</code>.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Progressive disclosure</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Frontmatter at startup, body on activation, supporting files only if the body points at them. Two lines of standing cost per skill, regardless of size.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>The description is the interface</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">True twice over — for a skill, and for a subagent in the <code style="font-size:15px; color:#2b8a3e;">task</code> tool. It's the only thing the caller sees when deciding. Say what it does <i>and</i> when to use it, in the words a request would use. Vague or overlapping descriptions are the main reason nothing fires.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Push detail down a level</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">If a fact is house-specific and load-bearing — a threshold, a matrix, a template — put it in <code style="font-size:15px; color:#2b8a3e;">references/</code> and tell the agent to read it rather than recall it. That instruction is what makes level 3 happen.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Subagents start empty</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">A custom subagent inherits <i>no</i> skills — pass <code style="font-size:15px; color:#2b8a3e;">skills</code> in its spec. That's a feature: a specialist's long runbook and its reference files stay out of the supervisor's context entirely, and the supervisor pays only for the subagent's one-line description.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><code style="font-size:15px; color:#2b8a3e;">memory=["/AGENTS.md", "/memory/notes.md"]</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">Injected every turn, no trigger. Keep it short. Multiple files concatenate in order, and HTML comments are stripped before injection.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Memory is writable</b></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The middleware instructs the agent to keep memory current with <code style="font-size:15px; color:#2b8a3e;">edit_file</code>. Give it a writable path and a correction sticks across threads, processes, and agents.</td>
</tr>
<tr>
</tbody>
</table>